# Fine-tune GPT-2 for email writing

Train a response-only LoRA adapter on synthetic email-writing pairs. The pretrained base model remains fixed. Select a GPU runtime before training. The repository includes an initial CPU-trained adapter; this notebook can train a larger run or continue from it.

Dataset: [Kamisori-daijin/email-datasets-20k](https://huggingface.co/datasets/Kamisori-daijin/email-datasets-20k), Apache 2.0 with a Gemma usage note. This is business-email data, not comprehensive letter-writing data.


In [ ]:
!git clone https://github.com/raoankit72005/minigpt.git /content/minigpt
%cd /content/minigpt
%pip install -r requirements.txt


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only; select a GPU before training")


## Try the included initial adapter
The first execution downloads the GPT-2 base checkpoint. Outputs may be repetitive or invent details; inspect before use.


In [ ]:
!python write_email.py --adapter email_adapter --request "Write a professional business email from a student to a professor regarding requesting a research meeting, specifically to discuss a machine learning project."


## Save training to Drive
Drive mounting is optional but recommended to retain artifacts after Colab disconnects. Training writes best/last adapters, tokenizers, metrics, and before/after samples.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT = "/content/drive/MyDrive/minigpt-email-lora"


## Fine-tune a larger run
Default example below uses up to 5,000 training emails, two epochs, and 200 records each for validation/test. Overlength emails are skipped rather than truncated. Prompt and padding labels are masked. Email purposes are held out by group.

Set TRAIN_EXAMPLES lower for an initial trial. To continue from the included adapter with a fresh optimizer, add `--resume-adapter email_adapter` to the command. This is not exact optimizer-state resumption.


In [ ]:
assert torch.cuda.is_available(), "Select a GPU before the larger training run."
import subprocess, sys
TRAIN_EXAMPLES = 5000
subprocess.run([sys.executable, "finetune_email.py", "--output", OUTPUT, "--train-examples", str(TRAIN_EXAMPLES), "--eval-examples", "200", "--epochs", "2", "--batch-size", "2", "--accumulation", "8", "--max-length", "512"], check=True)


## Inspect measured results
Lower held-out response perplexity is useful evidence but does not measure factual correctness or writing quality. Read samples and evaluate new, manually written requests too.


In [ ]:
import json
from pathlib import Path
results = json.loads((Path(OUTPUT)/"results.json").read_text())
print(json.dumps(results, indent=2))
samples = json.loads((Path(OUTPUT)/"samples.json").read_text())
for sample in samples:
    print("REQUEST:", sample["instruction"])
    print("BEFORE:", sample["before"])
    print("AFTER:", sample["after"])


## Generate with your trained adapter
Use the same Request/Email format as training.


In [ ]:
REQUEST = "Write a polite professional email requesting a project progress meeting next week. Mention that Tuesday afternoon works for me."
subprocess.run([sys.executable, "write_email.py", "--adapter", str(Path(OUTPUT)/"best"), "--request", REQUEST], check=True)


## Optional: merge into the custom GPT-2 implementation
This creates a large checkpoint with merged base/LoRA weights and checks output parity. Keep it in Drive, not Git.


In [ ]:
subprocess.run([sys.executable, "write_email.py", "--adapter", str(Path(OUTPUT)/"best"), "--request", REQUEST, "--export-custom", str(Path(OUTPUT)/"email-gpt2-custom.pt")], check=True)


## Download adapter and metrics
The adapter requires the original GPT-2 base model. The zip contains no base-model weights.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/minigpt-email-lora", "zip", OUTPUT)
files.download(archive)
